# 05 – Structure-grouped five-fold cold-drug cross-validation (retrained)
**Extends:** Section 5.1 and Supplementary Tables S5 (structure-purged re-evaluation) and S7 (structure-grouped retraining, reported here) of the revised manuscript. **Does not change** notebook `01_main_cold_drug.ipynb` or any shipped result.

**Why:** The main cold-drug folds partition the 309 compound *identifiers*. The structure-level audit (notebook 01, Cell 14c) found 20 compounds that are listed under two identifiers
(309 identifiers = 289 distinct structures), so in every fold 6-9 test compounds have a structural counterpart in the training or validation data. The manuscript handled this by removing
the affected test records *after* training. It could not remove the possible influence of such counterparts on the validation-based checkpoint selection, and it did not retrain on structure-grouped folds.
This notebook does exactly that:

1. **Group** all identifiers that denote the same compound (same criterion as the audit: same normalized name **or** same canonical SMILES of the largest fragment without stereochemistry; union-find).
2. **Partition the groups** (not the identifiers): shuffled once with seed 42 and cut into five disjoint sets; in every fold, 43 validation groups (15 % of all 289 groups, rounded down: int(0.15 × 289) = 43) are sampled from the groups that are not test groups. The record rule is unchanged
   (test = at least one test compound; validation = a validation compound and no test compound; training = the rest).
3. **Verify** that no identifier of a test group shares a name key or structure key with any identifier in that fold's training or validation data, and that validation and training are separated in the same way.
4. **Retrain** DualGATSynergy (same architecture, optimizer, schedule, checkpoint selection as notebook 01) and re-fit the Random Forest / XGBoost baselines, with and without the one-hot cell-line input, on these folds.

**What to compare:** the folds contain different compounds from the identifier-level folds, so compare **means over folds** (and the paired comparison with the baselines on the *same* new folds), not fold by fold.
The ablation, order-sensitivity and per-cohort analyses of the manuscript remain on the identifier-level folds; the main conclusions were already confirmed on purged test sets, and this notebook is the retraining confirmation.

**Limits that remain (state them in the manuscript):** the grouping is criterion-dependent (a different matching rule or RDKit version can change the number of duplicate compounds by one or two; the audit reported 20),
two *different* compounds with similar structures (e.g., analogues, salts not caught by the criterion) are still allowed in different folds (this is not a scaffold split), and each fold is trained once with one seed.

**How to run:** `Runtime → Run all` on a GPU runtime. Everything is **resumable**: after every fold the DualGATSynergy result row and test predictions, and after every fold the baseline rows and predictions, are written to Drive;
finished folds are skipped when you run the notebook again, so a disconnected session loses at most the fold in progress. Models are written to `checkpoints_retrained/structure_grouped/`.
Set `FOLDS_TO_RUN` (Cell 2) to run a subset. Standard deviations over folds use n−1.

In [ ]:
# ===== Cell 1: Environment =====
# Colab only. The package versions used for the reported results are pinned in requirements.txt.
# RDKit is also pinned here: its canonical SMILES (used for the structure-level audit in Cell 14c) can differ slightly between versions.
!pip install -q rdkit==2026.09.1 matplotlib pubchempy torch torchvision torchaudio torch-geometric transformers xgboost

# Standard library
import io, json, os, sys
from pathlib import Path

# Numerics and data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, ttest_rel

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset

# Graph learning
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.nn.aggr import AttentionalAggregation
from torch_geometric.utils import to_dense_batch

# Chemistry
from rdkit import Chem, RDLogger, Geometry
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import SimilarityMaps, rdMolDraw2D
from PIL import Image
from rdkit.Chem import Draw, rdDepictor


# Pretrained chemical language model (ChemBERTa)
from transformers import AutoTokenizer, AutoModel

# Classical ML and metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import average_precision_score, mean_squared_error, roc_auc_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

RDLogger.DisableLog("rdApp.*")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.8 MB/s eta 0:00:00


In [ ]:
# ===== Cell 2: Configuration =====
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/DualGATSynergy") if IN_COLAB else Path(".")   # same project folder as notebook 01
DATA_DIR = ROOT / "data"

FOLDS_TO_RUN  = [1, 2, 3, 4, 5]      # subset allowed, e.g. [1, 2]
RUN_DUALGAT   = True                 # retrain DualGATSynergy (GPU)
RUN_BASELINES = True                 # re-fit Random Forest / XGBoost (CPU, slow)

CKPT_DIR   = ROOT / "checkpoints_retrained" / "structure_grouped"
OUT_DIR    = ROOT / "results_structure_grouped"
ROWS_DIR   = OUT_DIR / "rows"            # one small CSV per finished fold and model family: this makes re-runs resumable
PREDS_DIR  = OUT_DIR / "predictions"
for d in (CKPT_DIR, ROWS_DIR, PREDS_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
def set_seed(seed=SEED):
    """Called before every fold, so a fold does not depend on what was executed before it."""
    torch.manual_seed(seed)   # also seeds CUDA
    np.random.seed(seed)
set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE} | project folder: {ROOT}")
if RUN_DUALGAT and DEVICE == "cpu":
    print("⚠ No GPU found: training will be very slow. Runtime → Change runtime type → GPU.")

Mounted at /content/drive
device: cuda | project folder: /content/drive/MyDrive/DualGATSynergy


## Data, molecular representations and the identifier-level partition (copied unchanged from notebook 01)

In [ ]:
# ===== Cell 3: Load and filter the raw data =====
# --- 3.1 DrugComb v1.5 ---
DRUGCOMB_COLS = ["drug_row", "drug_col", "cell_line_name", "synergy_hsa", "study_name", "tissue_name"]
drugcomb_raw = pd.read_csv(DATA_DIR / "drugcomb_full_v1.5.csv", usecols=DRUGCOMB_COLS, low_memory=False)
print(f"Raw records:                 {len(drugcomb_raw):,}   (paper: 1,432,351)")

# Remove non-oncology studies
NON_ONCOLOGY_KEYWORDS = ["malaria", "covid", "sars", "plasmodium", "tuberculosis"]
is_non_oncology = (drugcomb_raw["study_name"].astype(str).str.lower()
                   .str.contains("|".join(NON_ONCOLOGY_KEYWORDS), na=False))
drugcomb_raw = drugcomb_raw[~is_non_oncology].copy()

# Drop records without an HSA score or a cell-line name (removes nothing in v1.5)
drugcomb_raw = drugcomb_raw.dropna(subset=["synergy_hsa", "cell_line_name"])
n_compounds = len(set(drugcomb_raw["drug_row"]) | set(drugcomb_raw["drug_col"]))
print(f"After filtering:             {len(drugcomb_raw):,}   (paper: 1,432,146) | compounds: {n_compounds:,} (paper: 4,608)")

# --- 3.2 Compound -> SMILES cache (resolved through PubChem) ---
cache = pd.read_csv(DATA_DIR / "smiles_cache.csv")        # columns: name, smiles (empty if not resolved)
drug_to_smiles = dict(zip(cache["name"], cache["smiles"]))
known_drugs = {name for name, smi in drug_to_smiles.items() if isinstance(smi, str) and smi}
print(f"Cache: {len(drug_to_smiles)} names, {len(known_drugs)} resolved   (paper: 452 / 418)")

# Keep only records where BOTH compounds have a SMILES string
drugcomb = drugcomb_raw[drugcomb_raw["drug_row"].isin(known_drugs) &
                        drugcomb_raw["drug_col"].isin(known_drugs)].copy()
n_drugs = len(set(drugcomb["drug_row"]) | set(drugcomb["drug_col"]))
print(f"DrugComb subset: {len(drugcomb):,} records, {n_drugs} drugs   (paper: 61,111 / 276)")

# --- 3.3 O'Neil ---
oneil      = pd.read_csv(DATA_DIR / "oneil_summary_idx.csv")
drugs_meta = pd.read_csv(DATA_DIR / "drugs.csv")
cells_meta = pd.read_csv(DATA_DIR / "cell_lines.csv")
drug_map = dict(zip(drugs_meta["id"], drugs_meta["name"]))
cell_map = dict(zip(cells_meta["id"], cells_meta["name"]))
oneil["drug_row"]       = oneil["drug_row_idx"].map(drug_map)
oneil["drug_col"]       = oneil["drug_col_idx"].map(drug_map)
oneil["cell_line_name"] = oneil["cell_line_idx"].map(cell_map)
print(f"O'Neil: {len(oneil):,} records   (paper: 22,737) | unmapped names: "
      f"{int(oneil[['drug_row', 'drug_col', 'cell_line_name']].isna().sum().sum())}")

Raw records:                 1,432,351   (paper: 1,432,351)
After filtering:             1,432,146   (paper: 1,432,146) | compounds: 4,608 (paper: 4,608)
Cache: 452 names, 418 resolved   (paper: 452 / 418)
DrugComb subset: 61,111 records, 276 drugs   (paper: 61,111 / 276)
O'Neil: 22,737 records   (paper: 22,737) | unmapped names: 0


In [ ]:
# ===== Cell 4: O'Neil records with SMILES =====
oneil["smiles_row"] = oneil["drug_row"].map(drug_to_smiles)
oneil["smiles_col"] = oneil["drug_col"].map(drug_to_smiles)
oneil_processed = oneil.dropna(subset=["smiles_row", "smiles_col"])[
    ["drug_row", "drug_col", "smiles_row", "smiles_col", "cell_line_name", "synergy_loewe"]].copy()
oneil_processed.columns = ["drug_1", "drug_2", "smiles_1", "smiles_2", "cell_line", "synergy_score"]
print(f"O'Neil: {len(oneil_processed):,} of {len(oneil):,} records have SMILES for both drugs   (paper: 22,737 of 22,737)")

O'Neil: 22,737 of 22,737 records have SMILES for both drugs   (paper: 22,737 of 22,737)


In [ ]:
# ===== Cell 5: ChemBERTa embeddings (frozen pretrained model, [CLS] token) =====
CHEMBERTA_NAME       = "seyonec/ChemBERTa-zinc-base-v1"
CHEMBERTA_SHIPPED    = DATA_DIR / "drug_to_chemberta.pt"               # 418 compounds x 768 numbers (about 1.3 MB)
CHEMBERTA_RECOMPUTED = DATA_DIR / "drug_to_chemberta_recomputed.pt"    # never overwrites the shipped file
RECOMPUTE_CHEMBERTA  = False   # False: load the shipped file (no download, CPU is fine)
                               # True : recompute with the pretrained model (inference only, no training)

if not RECOMPUTE_CHEMBERTA:
    drug_to_chemberta = torch.load(CHEMBERTA_SHIPPED, map_location="cpu")
else:
    tokenizer = AutoTokenizer.from_pretrained(CHEMBERTA_NAME)
    chemberta = AutoModel.from_pretrained(CHEMBERTA_NAME).eval().to(DEVICE)
    print("Model revision:", getattr(chemberta.config, "_commit_hash", "unknown"))

    @torch.no_grad()
    def chemberta_embedding(smiles):
        """[CLS]-token embedding of one SMILES string (truncated to 128 tokens)."""
        inputs = tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
        return chemberta(**inputs).last_hidden_state[:, 0, :].squeeze(0).cpu()

    drug_to_chemberta = {name: chemberta_embedding(drug_to_smiles[name]) for name in sorted(known_drugs)}
    torch.save(drug_to_chemberta, CHEMBERTA_RECOMPUTED)

CHEMBERTA_DIM = next(iter(drug_to_chemberta.values())).shape[0]
print(f"ChemBERTa embeddings: {len(drug_to_chemberta)} compounds | dim = {CHEMBERTA_DIM}   (paper pipeline: 418 | 768)")
assert set(drug_to_chemberta) == known_drugs, "embedding file and SMILES cache do not match"

ChemBERTa embeddings: 418 compounds | dim = 768   (paper pipeline: 418 | 768)


In [ ]:
# ===== Cell 6: Molecular graphs =====
NODE_FEATURES = ["atomic_number", "degree", "explicit_valence", "implicit_valence", "is_aromatic", "in_ring",
                 "formal_charge", "num_radical_electrons", "hybridization", "chiral_tag", "atomic_mass"]
NODE_DIM, EDGE_DIM = len(NODE_FEATURES), 1      # edge feature: bond order (1.0, 1.5, 2.0 or 3.0)

def _valences(atom):
    """(explicit, implicit) valence. GetExplicitValence/GetImplicitValence produced the reported results;
    newer RDKit versions replace them with GetValence(...)."""
    try:
        return atom.GetExplicitValence(), atom.GetImplicitValence()
    except AttributeError:
        return atom.GetValence(Chem.ValenceType.EXPLICIT), atom.GetValence(Chem.ValenceType.IMPLICIT)

def smiles_to_graph(smiles):
    """SMILES string -> PyG graph (node features: NODE_FEATURES; edge feature: bond order).
    Returns None if the SMILES is missing or cannot be parsed."""
    if not isinstance(smiles, str) or not smiles:
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    features = []
    for atom in mol.GetAtoms():
        explicit_valence, implicit_valence = _valences(atom)
        features.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            explicit_valence,
            implicit_valence,
            int(atom.GetIsAromatic()),
            int(atom.IsInRing()),
            atom.GetFormalCharge(),
            atom.GetNumRadicalElectrons(),
            int(atom.GetHybridization()),
            int(atom.GetChiralTag()),       # RDKit chiral tag (none / clockwise / counter-clockwise / other)
            atom.GetMass(),
        ])
    x = torch.tensor(features, dtype=torch.float)

    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        s, e = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        order = bond.GetBondTypeAsDouble()
        edge_index += [(s, e), (e, s)]           # undirected: both directions
        edge_attr  += [order, order]

    if edge_index:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(edge_attr,  dtype=torch.float).view(-1, 1)
    else:                                         # single-atom molecule
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr  = torch.empty((0, 1),   dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

# Sanity check on all resolved compounds
graph_check = {name: smiles_to_graph(drug_to_smiles[name]) for name in known_drugs}
n_failed   = sum(g is None for g in graph_check.values())
n_no_bonds = sum(g is not None and g.edge_index.shape[1] == 0 for g in graph_check.values())
print(f"{len(graph_check)} cached compounds | failed to parse: {n_failed} | without any bond: {n_no_bonds} "
      f"(the paper's statement concerns the 309 compounds used in the experiments; see the check after Cell 7)")
assert all(g.x.shape[1] == NODE_DIM for g in graph_check.values() if g is not None)
del graph_check

418 cached compounds | failed to parse: 0 | without any bond: 2 (the paper's statement concerns the 309 compounds used in the experiments; see the check after Cell 7)


In [ ]:
# ===== Cell 7: Combine DrugComb (HSA) and O'Neil (Loewe) =====
drugcomb_unified = (drugcomb.dropna(subset=["synergy_hsa", "cell_line_name"])
                    [["drug_row", "drug_col", "synergy_hsa", "cell_line_name"]]
                    .rename(columns={"drug_row": "drug_1", "drug_col": "drug_2",
                                     "synergy_hsa": "raw_target", "cell_line_name": "cell_line"}))
drugcomb_unified["dataset_type"] = "drugcomb"

oneil_unified = (oneil_processed.dropna(subset=["synergy_score"])
                 [["drug_1", "drug_2", "synergy_score", "cell_line"]]
                 .rename(columns={"synergy_score": "raw_target"}))
oneil_unified["dataset_type"] = "oneil"

df_combined = pd.concat([drugcomb_unified, oneil_unified], ignore_index=True)

# One integer per cell line (order of first appearance) and the list of all compounds
all_cell_lines   = df_combined["cell_line"].unique()
cell_line_to_idx = {cl: i for i, cl in enumerate(all_cell_lines)}
n_cell_lines     = len(all_cell_lines)
all_drugs        = pd.concat([df_combined["drug_1"], df_combined["drug_2"]]).unique()

print(f"Records: {len(df_combined):,} (paper: 83,848) | cell-line identifiers: {n_cell_lines} (paper: 251) "
      f"| compounds: {len(all_drugs)} (paper: 309)")

Records: 83,848 (paper: 83,848) | cell-line identifiers: 251 (paper: 251) | compounds: 309 (paper: 309)


In [ ]:
# ===== Cell 8: Five-fold cold-drug partition =====
COHORTS = ["drugcomb", "oneil"]

def standardize_labels(train_df, val_df, test_df, fold):
    """Cohort-wise z-scoring of the synergy score into column 'label'.
    Mean and SD of each cohort come from the TRAINING rows only and are then applied to that cohort's
    validation and test rows, so nothing from validation or test enters the statistics."""
    scalers = {}
    for cohort in COHORTS:
        mask = train_df["dataset_type"] == cohort
        if mask.sum() == 0:
            print(f"  [warn] fold={fold}: no '{cohort}' samples in train -> scaler skipped")
            scalers[cohort] = None
            continue
        scalers[cohort] = StandardScaler()
        train_df.loc[mask, "label"] = scalers[cohort].fit_transform(train_df.loc[mask, ["raw_target"]])
    for split_df, split_name in ((val_df, "val"), (test_df, "test")):
        for cohort in COHORTS:
            mask = split_df["dataset_type"] == cohort
            if mask.sum() == 0:
                continue
            if scalers[cohort] is not None:
                split_df.loc[mask, "label"] = scalers[cohort].transform(split_df.loc[mask, ["raw_target"]])
            else:   # no training statistics for this cohort: its rows cannot be standardized, so they are dropped
                print(f"  [warn] fold={fold}: dropping {mask.sum()} '{cohort}' rows from {split_name} (no train scaler)")
                split_df.drop(split_df[mask].index, inplace=True)
    return scalers


def build_cold_drug_kfold_partition(df_combined, all_drugs, n_folds=5, val_frac=0.15, seed=42):
    """Drugs (not records) are partitioned. All drugs are shuffled once and cut into n_folds disjoint groups;
    each group is the test set of exactly one fold. Within a fold:
      test  = every record containing at least one test drug
      val   = records with a validation drug and no test drug (46 validation drugs, sampled from the non-test drugs)
      train = all other records (no test drug, no validation drug)
    """
    rng = np.random.RandomState(seed)
    shuffled_drugs = rng.permutation(all_drugs)
    test_groups = np.array_split(shuffled_drugs, n_folds)

    folds = []
    for fold_idx, test_drugs_arr in enumerate(test_groups):
        test_drugs = set(test_drugs_arr)
        remaining_drugs = np.array([d for d in all_drugs if d not in test_drugs])
        fold_rng = np.random.RandomState(seed * 1000 + fold_idx)
        n_val = int(val_frac * len(all_drugs))
        val_drugs = set(fold_rng.choice(remaining_drugs, size=min(n_val, len(remaining_drugs)), replace=False))

        is_test_pair  = df_combined["drug_1"].isin(test_drugs) | df_combined["drug_2"].isin(test_drugs)
        is_val_pair   = (~is_test_pair) & (df_combined["drug_1"].isin(val_drugs) | df_combined["drug_2"].isin(val_drugs))
        is_train_pair = ~is_test_pair & ~is_val_pair
        train_df, val_df, test_df = (df_combined[m].copy() for m in (is_train_pair, is_val_pair, is_test_pair))
        scalers = standardize_labels(train_df, val_df, test_df, fold_idx + 1)

        folds.append({"fold": fold_idx + 1, "train_df": train_df, "val_df": val_df, "test_df": test_df,
                      "test_drugs": test_drugs, "val_drugs": val_drugs,
                      "n_test_drugs": len(test_drugs), "n_val_drugs": len(val_drugs), "scalers": scalers})
    return folds


def verify_partition(folds, all_drugs):
    """Every drug must be in exactly one test set (no omissions, no duplicates)."""
    all_test_drugs = [d for f in folds for d in f["test_drugs"]]
    counts = pd.Series(all_test_drugs).value_counts()
    n_never, n_duplicated = len(all_drugs) - len(counts), int((counts > 1).sum())
    print(f"Test coverage: {len(counts)}/{len(all_drugs)} drugs | omitted: {n_never} | duplicated: {n_duplicated}")
    assert n_never == 0 and n_duplicated == 0, "not a true partition"
    print("✓ partition verified: every drug is in exactly one test set.")


def check_or_save_json(path, obj, what):
    """First run: save. Later runs (and reviewers): the rebuilt object must equal the shipped file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        assert json.loads(path.read_text(encoding="utf-8")) == obj, f"{what} differs from the shipped file {path.name}"
        print(f"✓ {what} identical to the shipped file")
    else:
        path.write_text(json.dumps(obj, indent=1, ensure_ascii=False), encoding="utf-8")
        print(f"Saved {what} -> {path}")


folds = build_cold_drug_kfold_partition(df_combined, all_drugs, n_folds=5, seed=42)
verify_partition(folds, all_drugs)
check_or_save_json(ROOT / "splits" / "cold_drug_5fold.json",
                   {str(f["fold"]): {"test_drugs": sorted(map(str, f["test_drugs"])),
                                     "val_drugs":  sorted(map(str, f["val_drugs"]))} for f in folds}, "cold-drug split")
check_or_save_json(ROOT / "splits" / "cell_line_to_idx.json",
                   {str(k): int(v) for k, v in cell_line_to_idx.items()}, "cell-line index")

print("\nRecords per partition (compare with Table 2):")
print(pd.DataFrame([{"fold": f["fold"], "train": len(f["train_df"]), "validation": len(f["val_df"]),
                     "test": len(f["test_df"])} for f in folds]).to_string(index=False))

Test coverage: 309/309 drugs | omitted: 0 | duplicated: 0
✓ partition verified: every drug is in exactly one test set.
✓ cold-drug split identical to the shipped file
✓ cell-line index identical to the shipped file

Records per partition (compare with Table 2):
 fold  train  validation  test
    1  31943       16459 35446
    2  44109       12321 27418
    3  31761       20452 31635
    4  37365       16643 29840
    5  33848       22116 27884


In [ ]:
# Keep the identifier-level folds of notebook 01 for the comparison at the end; `folds` is not used below.
identifier_folds = folds
del folds

## Model, dataset and training code (copied unchanged from notebook 01)

In [ ]:
# ===== Cell 9: Dataset =====
_mol_graph_cache = {}

def get_cached_graph(smiles):
    """Build each molecular graph once; return a copy so per-sample attributes never leak into the cache."""
    if smiles not in _mol_graph_cache:
        _mol_graph_cache[smiles] = smiles_to_graph(smiles)
    cached = _mol_graph_cache[smiles]
    return cached.clone() if cached is not None else None


class UnifiedSynergyDataset(Dataset):
    """Sample = (graph of drug 1, graph of drug 2, standardized synergy label).
    A record is kept only if both drugs have a ChemBERTa embedding and a SMILES that RDKit can parse."""
    def __init__(self, dataframe, drug_smiles_dict, drug_chemberta_dict, cell_line_to_idx):
        self.data        = dataframe.reset_index(drop=True)
        self.drug_smiles = drug_smiles_dict
        self.drug_cb     = drug_chemberta_dict
        self.cell_to_idx = cell_line_to_idx

        valid_drugs = {name for name in drug_chemberta_dict
                       if isinstance(drug_smiles_dict.get(name), str)
                       and Chem.MolFromSmiles(drug_smiles_dict[name]) is not None}
        keep = self.data["drug_1"].isin(valid_drugs) & self.data["drug_2"].isin(valid_drugs)
        self.valid_indices = list(self.data.index[keep])
        print(f"Valid samples: {len(self.valid_indices):,} of {len(self.data):,}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        row = self.data.iloc[self.valid_indices[idx]]
        g1 = get_cached_graph(self.drug_smiles[row["drug_1"]])
        g2 = get_cached_graph(self.drug_smiles[row["drug_2"]])
        g1.cb = self.drug_cb[row["drug_1"]].unsqueeze(0)
        g2.cb = self.drug_cb[row["drug_2"]].unsqueeze(0)
        cl_idx = self.cell_to_idx[row["cell_line"]]
        g1.cell_idx = torch.tensor([cl_idx], dtype=torch.long)
        g2.cell_idx = torch.tensor([cl_idx], dtype=torch.long)
        target = torch.tensor(row["label"], dtype=torch.float)
        return g1, g2, target

In [ ]:
# ===== Cell 10: DualGATSynergy =====
class DualGATSynergy(nn.Module):
    def __init__(self, num_atom_features, chemberta_dim, n_cell_lines, hidden_dim=64, heads=2):
        super().__init__()
        # Weight-shared two-layer GATv2 atom encoder (bond order is the edge feature)
        self.conv1 = GATv2Conv(num_atom_features, hidden_dim, heads=heads, edge_dim=1, dropout=0.2)
        self.conv2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, concat=False, edge_dim=1, dropout=0.2)

        # Gated-attention pooling (one gate network, used for all four pooled streams)
        self.pool = AttentionalAggregation(gate_nn=nn.Sequential(
            nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Linear(32, 1)))

        # IMCA: 4-head atom-level cross-attention; the same weights are used in both directions
        self.cross_attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=4, dropout=0.1, batch_first=True)

        # Projection of the 768-d ChemBERTa vector; LayerNorm aligns its scale with the GAT vectors
        self.chemberta_proj = nn.Sequential(
            nn.Linear(chemberta_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(0.3))

        self.cell_embedding = nn.Embedding(n_cell_lines, hidden_dim)

        # 7 streams x 64 = 448 inputs
        self.predictor = nn.Sequential(
            nn.Linear(hidden_dim * 7, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1))

    def encode_atoms(self, x, edge_index, edge_attr):
        x = F.elu(self.conv1(x, edge_index, edge_attr=edge_attr))
        x = F.dropout(x, p=0.2, training=self.training)
        return self.conv2(x, edge_index, edge_attr=edge_attr)

    def forward(self, d1, d2):
        atoms1 = self.encode_atoms(d1.x, d1.edge_index, d1.edge_attr)     # per-atom embeddings, before any pooling
        atoms2 = self.encode_atoms(d2.x, d2.edge_index, d2.edge_attr)
        emb1 = self.pool(atoms1, d1.batch)
        emb2 = self.pool(atoms2, d2.batch)

        dense1, mask1 = to_dense_batch(atoms1, d1.batch)                   # pad molecules to equal size
        dense2, mask2 = to_dense_batch(atoms2, d2.batch)
        attn_1to2, w_1to2 = self.cross_attention(dense1, dense2, dense2, key_padding_mask=~mask2)
        attn_2to1, w_2to1 = self.cross_attention(dense2, dense1, dense1, key_padding_mask=~mask1)
        pooled_1to2 = self.pool(attn_1to2[mask1], d1.batch)                # drop padded atoms, then pool
        pooled_2to1 = self.pool(attn_2to1[mask2], d2.batch)

        cb1_proj = self.chemberta_proj(d1.cb)
        cb2_proj = self.chemberta_proj(d2.cb)
        cell_emb = self.cell_embedding(d1.cell_idx)

        combined = torch.cat([emb1, emb2, pooled_1to2, pooled_2to1, cb1_proj, cb2_proj, cell_emb], dim=1)
        return self.predictor(combined), (w_1to2, w_2to1)


MODEL_CFG = dict(num_atom_features=NODE_DIM, chemberta_dim=CHEMBERTA_DIM, n_cell_lines=n_cell_lines, hidden_dim=64)
_m = DualGATSynergy(**MODEL_CFG)
n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}   (paper: 260,482)")
assert n_params == 260_482
del _m

Trainable parameters: 260,482   (paper: 260,482)


In [ ]:
# ===== Cell 11: Training and evaluation functions =====
def evaluate_model(model, loader):
    """RMSE, PCC and the raw predictions / targets of `model` on `loader`."""
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in loader:
            d1, d2, target = batch[0].to(DEVICE), batch[1].to(DEVICE), batch[2].to(DEVICE)
            pred, _ = model(d1, d2)
            all_preds.extend(pred.cpu().numpy().flatten())
            all_targets.extend(target.cpu().numpy().flatten())
    rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
    pcc, _ = pearsonr(all_targets, all_preds)
    return float(rmse), float(pcc), all_preds, all_targets


def train_one_fold(model, train_loader, val_loader, checkpoint_path, num_epochs=200, patience_epochs=40):
    """Adam + Smooth-L1. ONLY the validation set drives the LR scheduler, early stopping and checkpoint selection
    (best validation PCC); the test set is never used here. Returns the best validation PCC."""
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
    criterion = nn.SmoothL1Loss()
    best_val_pcc, epochs_since_improve = -1.0, 0

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            d1, d2, targets = batch[0].to(DEVICE), batch[1].to(DEVICE), batch[2].to(DEVICE)
            optimizer.zero_grad()
            predictions, _ = model(d1, d2)
            loss = criterion(predictions, targets.view(-1, 1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)

        if (epoch + 1) % 2 == 0:                          # validate every 2 epochs
            val_rmse, val_pcc, _, _ = evaluate_model(model, val_loader)
            scheduler.step(val_pcc)
            print(f"  Epoch {epoch+1:03d} | Loss: {avg_loss:.4f} | Val RMSE: {val_rmse:.4f} | Val PCC: {val_pcc:.4f}")
            if val_pcc > best_val_pcc:
                best_val_pcc, epochs_since_improve = val_pcc, 0
                torch.save(model.state_dict(), checkpoint_path)
            else:
                epochs_since_improve += 2
            if epochs_since_improve >= patience_epochs:
                print(f"  Early stopping at epoch {epoch+1}")
                break
    return best_val_pcc

## Structure groups and the structure-grouped partition

In [ ]:
# ===== Cell 12: Structure groups (the grouping step of the audit, Cell 14c of notebook 01, unchanged) =====
# DrugComb and O'Neil name compounds differently (common names, salt forms, CAS numbers, ChEMBL IDs, internal codes), so one molecule
# can occur under two identifiers. The cold-drug partition is by identifier: a molecule can therefore be a test compound under one
# name and be present in the training or validation data under another. This cell
#   (1) finds the compounds that occur under more than one identifier,
#   (2) marks, in each fold, the test compounds that have such a twin in that fold's training or validation data,
#   (3) removes the test records that contain one of them and re-evaluates DualGATSynergy and the baselines on the remaining
#       ("purged") test records. Nothing is retrained: the predictions of Cells 12, 14 and 14b are re-used.
# Two identifiers denote the same compound if they have the same canonical SMILES of the largest fragment (counter-ions removed,
# stereochemistry removed) OR the same name after lower-casing and deleting everything except letters and digits.
import re
import rdkit
from rdkit.Chem.MolStandardize import rdMolStandardize

print(f"RDKit {rdkit.__version__}   (version used for the reported results: 2026.09.1). A different version can canonicalize a few "
      f"structures differently and change the number of duplicate compounds by one or two.")

_largest_fragment = rdMolStandardize.LargestFragmentChooser()

def structure_key(name):
    """Canonical SMILES of the largest fragment (counter-ions removed), without stereochemistry."""
    smi = drug_to_smiles.get(name)
    mol = Chem.MolFromSmiles(smi) if isinstance(smi, str) and smi else None
    return Chem.MolToSmiles(_largest_fragment.choose(mol), isomericSmiles=False) if mol is not None else None

def identity_keys(name):
    keys = {("name", re.sub(r"[^a-z0-9]", "", str(name).lower()))}
    smi = structure_key(name)
    if smi:
        keys.add(("structure", smi))
    return keys

KEYS = {d: identity_keys(d) for d in all_drugs}

# --- (1) Groups of identifiers that denote the same compound (union-find over shared keys) ---
_parent = {d: d for d in all_drugs}
def _find(x):
    while _parent[x] != x:
        _parent[x] = _parent[_parent[x]]
        x = _parent[x]
    return x
_first_owner = {}
for d in all_drugs:
    for k in KEYS[d]:
        if k in _first_owner:
            _parent[_find(d)] = _find(_first_owner[k])
        else:
            _first_owner[k] = d
_groups = {}
for d in all_drugs:
    _groups.setdefault(_find(d), []).append(d)
duplicate_groups = sorted(sorted(map(str, g)) for g in _groups.values() if len(g) > 1)
print(f"Identifiers: {len(all_drugs)} | distinct compounds: {len(_groups)} | compounds listed under two identifiers: {len(duplicate_groups)}"
      f"   (paper: 309 / 289 / 20)")
for g in duplicate_groups:
    print("   ", g)

dup_path = ROOT / "splits" / "duplicate_compounds.json"
if dup_path.exists():
    same = json.loads(dup_path.read_text(encoding="utf-8")) == duplicate_groups
    print("✓ duplicate groups identical to the shipped file" if same else
          f"⚠ duplicate groups differ from the shipped file {dup_path.name} (RDKit versions can canonicalize a few structures differently)")
else:
    dup_path.parent.mkdir(parents=True, exist_ok=True)
    dup_path.write_text(json.dumps(duplicate_groups, indent=1, ensure_ascii=False), encoding="utf-8")
    print(f"Saved duplicate groups -> {dup_path}")



RDKit 2026.09.1   (version used for the reported results: 2026.09.1). A different version can canonicalize a few structures differently and change the number of duplicate compounds by one or two.
Identifiers: 309 | distinct compounds: 289 | compounds listed under two identifiers: 20   (paper: 309 / 289 / 20)
    ['122111-05-1', 'GEMCITABINE']
    ['14913-33-8', 'cis-Platin']
    ['34031-32-8', 'AURANOFIN']
    ['391210-10-9', 'PD325901']
    ['891494-63-6', 'MK-8776']
    ['915019-65-7', 'BEZ-235']
    ['ABT-888', 'Veliparib']
    ['ADM hydrochloride', 'DOXORUBICIN']
    ['AZD1775', 'MK-1775']
    ['CHEMBL3348822', 'NSC733504']
    ['CHEMBL3348857', 'ETOPOSIDE']
    ['CYCLOPHOSPHAMIDE', 'cyclophosphamide']
    ['DASATINIB', 'Dasatinib']
    ['MRK-003', 'MRK003']
    ['Navelbine ditartrate (TN)', 'VINORELBINE']
    ['SUNITINIB', 'Sunitinib malate']
    ['TOPOTECAN', 'TOPOTECAN HYDROCHLORIDE']
    ['Tamoxifen citrate', 'tamoxifen']
    ['Vincristine sulfate', 'vincristine']
    ['Zanosar

In [ ]:
# ===== Cell 13: Structure-grouped five-fold cold-drug partition =====
order_of = {d: i for i, d in enumerate(all_drugs)}
groups = sorted((sorted(g, key=order_of.get) for g in _groups.values()), key=lambda g: order_of[g[0]])   # deterministic order
n_groups = len(groups)
print(f"{len(all_drugs)} identifiers -> {n_groups} groups (paper: 309 -> 289); groups with two identifiers: "
      f"{sum(len(g) > 1 for g in groups)} (paper: 20); largest group: {max(len(g) for g in groups)} identifiers")

def index_by_key(drugs):
    index = {}
    for d in drugs:
        for k in KEYS[d]:
            index.setdefault(k, set()).add(d)
    return index

def twins_in(drug, index):
    """Other identifiers of the same compound among the drugs of `index`."""
    return set().union(*[index.get(k, set()) for k in KEYS[drug]]) - {drug}

def build_structure_grouped_kfold_partition(df_combined, groups, n_folds=5, val_frac=0.15, seed=42):
    """Like build_cold_drug_kfold_partition (Cell 8), but GROUPS of identifiers are partitioned, so all identifiers of one compound
    are in the same fold and the same role. Groups are shuffled once and cut into n_folds disjoint sets; each set is the test set of one
    fold. Within a fold: validation groups = val_frac of ALL groups, sampled from the non-test groups; the record rule is the same:
      test  = every record containing at least one test compound
      val   = records with a validation compound and no test compound
      train = all other records
    """
    n = len(groups)
    rng = np.random.RandomState(seed)
    test_group_sets = np.array_split(rng.permutation(n), n_folds)
    folds = []
    for fold_idx, tg in enumerate(test_group_sets):
        tg_set = set(int(g) for g in tg)
        remaining = np.array([g for g in range(n) if g not in tg_set])
        fold_rng = np.random.RandomState(seed * 1000 + fold_idx)
        vg = fold_rng.choice(remaining, size=min(int(val_frac * n), len(remaining)), replace=False)
        test_drugs = {d for g in tg_set for d in groups[g]}
        val_drugs  = {d for g in vg for d in groups[int(g)]}

        is_test_pair  = df_combined["drug_1"].isin(test_drugs) | df_combined["drug_2"].isin(test_drugs)
        is_val_pair   = (~is_test_pair) & (df_combined["drug_1"].isin(val_drugs) | df_combined["drug_2"].isin(val_drugs))
        is_train_pair = ~is_test_pair & ~is_val_pair
        train_df, val_df, test_df = (df_combined[m].copy() for m in (is_train_pair, is_val_pair, is_test_pair))
        scalers = standardize_labels(train_df, val_df, test_df, fold_idx + 1)      # cohort-wise, training statistics only (Cell 8)
        folds.append({"fold": fold_idx + 1, "train_df": train_df, "val_df": val_df, "test_df": test_df,
                      "test_drugs": test_drugs, "val_drugs": val_drugs,
                      "n_test_groups": len(tg_set), "n_val_groups": len(vg),
                      "n_test_drugs": len(test_drugs), "n_val_drugs": len(val_drugs), "scalers": scalers})
    return folds

sfolds = build_structure_grouped_kfold_partition(df_combined, groups, n_folds=5, seed=42)

# ---- Verification ----
verify_partition(sfolds, all_drugs)          # every identifier is in exactly one test set (no omissions, no duplicates)
test_groups_seen = [frozenset(g) for f in sfolds for g in groups if set(g) <= f["test_drugs"]]
assert len(test_groups_seen) == n_groups and len(set(test_groups_seen)) == n_groups, "a group is split across folds or tested twice"
print("✓ every group is in exactly one test set (aliases are never separated)")

rows = []
for f in sfolds:
    k = f["fold"]
    tr, va, te = f["train_df"], f["val_df"], f["test_df"]
    assert len(tr) + len(va) + len(te) == len(df_combined), f"fold {k}: partitions do not cover the data"
    assert not (set(tr.index) & set(va.index)) and not (set(tr.index) & set(te.index)) and not (set(va.index) & set(te.index)), "overlapping records"
    train_idx = index_by_key(set(tr["drug_1"]) | set(tr["drug_2"]))
    val_idx   = index_by_key(set(va["drug_1"]) | set(va["drug_2"]))
    n_test_twins = sum(bool(twins_in(d, train_idx) or twins_in(d, val_idx)) for d in f["test_drugs"])
    n_val_twins  = sum(bool(twins_in(d, train_idx)) for d in f["val_drugs"])
    assert n_test_twins == 0, f"fold {k}: {n_test_twins} test compounds have a structural/name counterpart in training or validation data"
    assert n_val_twins == 0, f"fold {k}: {n_val_twins} validation compounds have a counterpart in the training data"
    rows.append({"fold": k, "test groups": f["n_test_groups"], "test ids": f["n_test_drugs"], "val groups": f["n_val_groups"],
                 "val ids": f["n_val_drugs"], "train": len(tr), "validation": len(va), "test": len(te),
                 "test with twin in train/val": n_test_twins})
print("✓ no test compound has a name or structure counterpart in the training or validation data of its fold; "
      "no validation compound has one in the training data (checked with the audit's matching keys)")
print("\nRecords per partition (identifier-level folds of notebook 01 for comparison: see Table 2):")
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nTest evaluations in total: {sum(len(f['test_df']) for f in sfolds):,} "
      f"= {sum(len(f['test_df']) for f in sfolds) / len(df_combined):.2f} per record (a record is tested in every fold that holds out one of its compounds)")

check_or_save_json(ROOT / "splits" / "structure_grouped_5fold.json",
                   {str(f["fold"]): {"test_drugs": sorted(map(str, f["test_drugs"])),
                                     "val_drugs":  sorted(map(str, f["val_drugs"]))} for f in sfolds}, "structure-grouped split")
check_or_save_json(ROOT / "splits" / "structure_groups.json", [list(map(str, g)) for g in groups], "structure groups")

309 identifiers -> 289 groups (paper: 309 -> 289); groups with two identifiers: 20 (paper: 20); largest group: 2 identifiers
Test coverage: 309/309 drugs | omitted: 0 | duplicated: 0
✓ partition verified: every drug is in exactly one test set.
✓ every group is in exactly one test set (aliases are never separated)
✓ no test compound has a name or structure counterpart in the training or validation data of its fold; no validation compound has one in the training data (checked with the audit's matching keys)

Records per partition (identifier-level folds of notebook 01 for comparison: see Table 2):
 fold  test groups  test ids  val groups  val ids  train  validation  test  test with twin in train/val
    1           58        66          43       46  29232       16938 37678                            0
    2           58        60          43       46  33561       23272 27015                            0
    3           58        63          43       46  30885       17532 35431           

## Retrain DualGATSynergy on the structure-grouped folds (resumable)

In [ ]:
# ===== Cell 14: DualGATSynergy, one training run per fold; the test set is used exactly once per fold =====
def make_dataset(df):
    return UnifiedSynergyDataset(df, drug_to_smiles, drug_to_chemberta, cell_line_to_idx)

PRED_COLS = ["drug_1", "drug_2", "cell_line", "dataset_type", "raw_target", "label"]
dl_row_path  = lambda k: ROWS_DIR  / f"dualgat__fold{k}.csv"
dl_pred_path = lambda k: PREDS_DIR / f"dualgat__fold{k}.csv.gz"

if RUN_DUALGAT:
    for f in sfolds:
        k = f["fold"]
        if k not in FOLDS_TO_RUN:
            continue
        if dl_row_path(k).exists() and dl_pred_path(k).exists():
            print(f"⏭  DualGATSynergy | fold {k}: already done")
            continue
        print(f"\n{'='*60}\nFOLD {k}/5 | test groups: {f['n_test_groups']} ({f['n_test_drugs']} identifiers) | "
              f"validation groups: {f['n_val_groups']} ({f['n_val_drugs']} identifiers)\n{'='*60}")
        set_seed(SEED)
        model = DualGATSynergy(**MODEL_CFG).to(DEVICE)
        test_ds      = make_dataset(f["test_df"])
        test_loader  = DataLoader(test_ds, batch_size=128, shuffle=False)
        train_loader = DataLoader(make_dataset(f["train_df"]), batch_size=128, shuffle=True)
        val_loader   = DataLoader(make_dataset(f["val_df"]),   batch_size=128, shuffle=False)
        ckpt = CKPT_DIR / f"best_model_fold{k}.pth"
        best_val_pcc = train_one_fold(model, train_loader, val_loader, ckpt)          # validation set only

        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))                  # best-validation checkpoint
        test_rmse, test_pcc, test_preds, test_targets = evaluate_model(model, test_loader)   # the only use of the test set

        kept = f["test_df"].iloc[test_ds.valid_indices]
        assert np.allclose(kept["label"].values, test_targets, atol=1e-5), "rows and labels are misaligned"
        labels_cls = (kept["raw_target"].values > 10).astype(int)                     # positive: raw score > 10 (cohort's native scale)
        auc, aupr = roc_auc_score(labels_cls, test_preds), average_precision_score(labels_cls, test_preds)

        # predictions first, the result row LAST: a fold counts as finished only when its row exists; temporary names + rename
        tmp_p = dl_pred_path(k).with_name(dl_pred_path(k).name + ".tmp")
        kept[PRED_COLS].assign(pred=test_preds, fold=k).to_csv(tmp_p, index=False, compression="gzip")
        os.replace(tmp_p, dl_pred_path(k))
        row = {"fold": k, "n_test_groups": f["n_test_groups"], "n_test_drugs": f["n_test_drugs"], "val_pcc": best_val_pcc,
               "test_rmse": test_rmse, "test_pcc": test_pcc, "auc": auc, "aupr": aupr}
        tmp_r = dl_row_path(k).with_suffix(".tmp")
        pd.DataFrame([row]).to_csv(tmp_r, index=False)
        os.replace(tmp_r, dl_row_path(k))
        print(f"FOLD {k} -> Test RMSE: {test_rmse:.4f} | Test PCC: {test_pcc:.4f} | AUC: {auc:.4f} | AUPR: {aupr:.4f}   (saved)")

        del model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
else:
    print("RUN_DUALGAT = False: skipped (existing rows are still summarized in Cell 16).")

⏭  DualGATSynergy | fold 1: already done
⏭  DualGATSynergy | fold 2: already done
⏭  DualGATSynergy | fold 3: already done
⏭  DualGATSynergy | fold 4: already done
⏭  DualGATSynergy | fold 5: already done


## Fingerprint baselines on the structure-grouped folds, with and without cell-line input (resumable)

In [ ]:
# ===== Cell 15: Baselines (definitions copied unchanged from Cells 14 and 14b of notebook 01; the loops below are resumable) =====
def get_morgan_fingerprint(smiles, radius=2, n_bits=1024):
    """Morgan bit vector (radius 2, 1024 bits)."""
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) and smiles else None
    if mol is None:
        return None
    try:     # legacy RDKit API, used for the reported results
        return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits))
    except AttributeError:   # removed in newer RDKit: equivalent generator API
        from rdkit.Chem import rdFingerprintGenerator
        return rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits).GetFingerprintAsNumPy(mol)

drug_to_fp = {}
for name in sorted(known_drugs):
    fp = get_morgan_fingerprint(drug_to_smiles[name])
    if fp is not None:
        drug_to_fp[name] = fp

def fp_rows(df):
    """The records that have a fingerprint for both drugs (all of them here), in their original order."""
    return df[df["drug_1"].isin(drug_to_fp) & df["drug_2"].isin(drug_to_fp)]

def build_fp_dataset(df):
    """Features = [fingerprint of drug 1 | fingerprint of drug 2] (2,048 bits). No cell-line input."""
    d = fp_rows(df)
    X = np.hstack([np.stack(d["drug_1"].map(drug_to_fp).values), np.stack(d["drug_2"].map(drug_to_fp).values)])
    return X, d["label"].values, d["raw_target"].values

BASELINES = {
    "rf":  lambda: RandomForestRegressor(n_estimators=300, max_depth=12, max_features="sqrt", n_jobs=-1, random_state=42),
    "xgb": lambda: XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1),
}

PRED_ID_COLS = ["drug_1", "drug_2", "cell_line", "dataset_type", "raw_target", "label"]   # identifies each test prediction


# Same fixed Random Forest / XGBoost settings and the same five folds as Cell 14. The only change: a one-hot encoding of the
# cell line is appended to the two fingerprints (2,048 + n_cell_lines features). DualGATSynergy receives the cell line through an
# embedding, so this tests whether the gap to the baselines of Cell 14 comes from the missing cell-line input rather than from the architecture.
def build_fp_cell_dataset(df):
    """Features = [fingerprint of drug 1 | fingerprint of drug 2 | one-hot cell line]. Also returns the records that were kept."""
    d = fp_rows(df)
    fp = np.hstack([np.stack(d["drug_1"].map(drug_to_fp).values),
                    np.stack(d["drug_2"].map(drug_to_fp).values)]).astype(np.float32)     # float32: halves the memory, same trees
    onehot = np.zeros((len(d), n_cell_lines), dtype=np.float32)
    onehot[np.arange(len(d)), d["cell_line"].map(cell_line_to_idx).values.astype(int)] = 1.0
    return np.hstack([fp, onehot]), d["label"].values, d



bl_row_path  = lambda k: ROWS_DIR  / f"baselines__fold{k}.csv"
bl_pred_path = lambda k: PREDS_DIR / f"baselines__fold{k}.csv.gz"

if RUN_BASELINES:
    for f in sfolds:
        k = f["fold"]
        if k not in FOLDS_TO_RUN:
            continue
        if bl_row_path(k).exists() and bl_pred_path(k).exists():
            print(f"⏭  baselines | fold {k}: already done")
            continue
        X_tr,  y_tr,  _      = build_fp_dataset(f["train_df"])              # fingerprints only
        X_te,  y_te,  raw_te = build_fp_dataset(f["test_df"])
        Xc_tr, yc_tr, _      = build_fp_cell_dataset(f["train_df"])         # fingerprints + one-hot cell line
        Xc_te, yc_te, rows_c = build_fp_cell_dataset(f["test_df"])
        assert np.allclose(y_te, yc_te), "test rows differ between the two baseline inputs"
        labels_cls = (raw_te > 10).astype(int)
        row, fold_preds = {"fold": k}, fp_rows(f["test_df"])[PRED_ID_COLS].assign(fold=k)
        for key, make_model in BASELINES.items():
            for tag, (Xa, ya, Xb) in {"": (X_tr, y_tr, X_te), "_cell": (Xc_tr, yc_tr, Xc_te)}.items():
                preds = make_model().fit(Xa, ya).predict(Xb)
                name = f"{key}{tag}"
                fold_preds[name]      = preds
                row[f"{name}_pcc"]    = float(pearsonr(y_te, preds)[0])
                row[f"{name}_rmse"]   = float(np.sqrt(mean_squared_error(y_te, preds)))
                row[f"{name}_auc"]    = float(roc_auc_score(labels_cls, preds))
                row[f"{name}_aupr"]   = float(average_precision_score(labels_cls, preds))
        tmp_p = bl_pred_path(k).with_name(bl_pred_path(k).name + ".tmp")
        fold_preds.to_csv(tmp_p, index=False, compression="gzip")
        os.replace(tmp_p, bl_pred_path(k))
        tmp_r = bl_row_path(k).with_suffix(".tmp")
        pd.DataFrame([row]).to_csv(tmp_r, index=False)
        os.replace(tmp_r, bl_row_path(k))
        print(f"Fold {k} | RF {row['rf_pcc']:.4f} | XGB {row['xgb_pcc']:.4f} | RF+cell {row['rf_cell_pcc']:.4f} | XGB+cell {row['xgb_cell_pcc']:.4f}   (saved)")
else:
    print("RUN_BASELINES = False: skipped (existing rows are still summarized in Cell 16).")

⏭  baselines | fold 1: already done
⏭  baselines | fold 2: already done
⏭  baselines | fold 3: already done
⏭  baselines | fold 4: already done
Fold 5 | RF 0.2356 | XGB 0.1819 | RF+cell 0.2508 | XGB+cell 0.2666   (saved)


## Summary and comparison with the identifier-level folds

In [ ]:
# ===== Cell 16: Summary =====
def load_rows(pattern):
    paths = sorted(ROWS_DIR.glob(pattern))
    return pd.concat([pd.read_csv(p) for p in paths], ignore_index=True).sort_values("fold").reset_index(drop=True) if paths else None

gat = load_rows("dualgat__fold*.csv")
bl  = load_rows("baselines__fold*.csv")
assert gat is not None, "no DualGATSynergy folds finished yet (Cell 14)"
print(f"DualGATSynergy folds finished: {gat['fold'].tolist()} | baseline folds finished: {bl['fold'].tolist() if bl is not None else 'none'}\n")
print(gat.drop(columns=["val_pcc"]).round(4).to_string(index=False))

# Values of the identifier-level folds, reported in the manuscript (mean ± SD of the PCC over five folds; AUC mean)
PAPER = {"DualGATSynergy": ("0.346 ± 0.073", 0.656), "RF": ("0.261 ± 0.058", None), "XGB": ("0.207 ± 0.039", None),
         "RF+cell": ("0.282 ± 0.066", 0.636), "XGB+cell": ("0.333 ± 0.033", 0.633)}
COLS = {"DualGATSynergy": (gat, "test_pcc", "auc", "aupr"), "RF": (bl, "rf_pcc", "rf_auc", "rf_aupr"), "XGB": (bl, "xgb_pcc", "xgb_auc", "xgb_aupr"),
        "RF+cell": (bl, "rf_cell_pcc", "rf_cell_auc", "rf_cell_aupr"), "XGB+cell": (bl, "xgb_cell_pcc", "xgb_cell_auc", "xgb_cell_aupr")}

print("\nStructure-grouped folds, mean ± SD over the finished folds (n−1)   |   identifier-level folds (manuscript)")
summary = []
for name, (df, p, a, ap) in COLS.items():
    if df is None:
        continue
    summary.append({"model": name, "folds": len(df), "PCC mean": df[p].mean(), "PCC SD": df[p].std(),
                    "AUC": df[a].mean(), "AUPR": df[ap].mean()})
    print(f"{name:15s} PCC {df[p].mean():.3f} ± {df[p].std():.3f} | AUC {df[a].mean():.3f} | AUPR {df[ap].mean():.3f}"
          f"   |   PCC {PAPER[name][0]}" + (f" | AUC {PAPER[name][1]:.3f}" if PAPER[name][1] else ""))
pd.DataFrame(summary).to_csv(OUT_DIR / "summary_structure_grouped.csv", index=False)

# Paired comparison with the baselines on the SAME new folds (indicative: the folds share training data)
if bl is not None and len(gat) == len(bl) and (gat["fold"].values == bl["fold"].values).all() and len(gat) >= 3:
    print("\nPaired comparison, DualGATSynergy vs baseline on the same structure-grouped folds:")
    for name in ("RF", "XGB", "RF+cell", "XGB+cell"):
        col = COLS[name][1]
        diff = gat["test_pcc"].values - bl[col].values
        t, pval = ttest_rel(gat["test_pcc"], bl[col])
        print(f"vs {name:9s} mean PCC gain {diff.mean():+.3f} | t = {t:.2f} | p = {pval:.4f} | better in {(diff > 0).sum()}/{len(diff)} folds")
    print("(identifier-level folds, manuscript: RF gain 0.085, XGBoost gain 0.139, RF+cell gain 0.064, XGBoost+cell gain 0.013 (p = 0.63))")
else:
    print("\nThe paired comparison needs the same finished folds for DualGATSynergy and the baselines.")

# Per-cohort PCC of DualGATSynergy and within-cell-line PCC (same definition as in notebook 02b; check it against Table 11 there)
def within_cell_pcc(df, pred_col="pred", min_rows=30):
    vals, wts = [], []
    for _, g in df.groupby("cell_line"):
        if len(g) >= min_rows and g["label"].std() > 0 and g[pred_col].std() > 0:
            vals.append(pearsonr(g["label"], g[pred_col])[0]); wts.append(len(g))
    return float(np.average(vals, weights=wts)) if vals else float("nan")

paths = sorted(PREDS_DIR.glob("dualgat__fold*.csv.gz"))
if paths:
    dp = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    print(f"\nDualGATSynergy on the structure-grouped folds ({len(dp):,} test evaluations, folds {sorted(int(x) for x in dp['fold'].unique())}):")
    for cohort in ("drugcomb", "oneil"):
        sub = dp[dp["dataset_type"] == cohort]
        per_fold = pd.Series({int(k_): pearsonr(g_["label"], g_["pred"])[0] for k_, g_ in sub.groupby("fold")})
        print(f"  {cohort:9s} mean per-fold PCC {per_fold.mean():.3f} ± {per_fold.std():.3f} | within-cell-line PCC (pooled) {within_cell_pcc(sub):.3f}")
    print("  (identifier-level folds, Table 11: DrugComb 0.370 ± 0.075, within-cell 0.192 | O'Neil 0.281 ± 0.046, within-cell 0.289)")

DualGATSynergy folds finished: [1, 2, 3, 4, 5] | baseline folds finished: [1, 2, 3, 4, 5]

 fold  n_test_groups  n_test_drugs  test_rmse  test_pcc    auc   aupr
    1             58            66     0.9136    0.2921 0.6393 0.3438
    2             58            60     1.0118    0.2841 0.6245 0.3478
    3             58            63     0.8889    0.3438 0.6041 0.2943
    4             58            59     0.9487    0.3358 0.6265 0.2277
    5             57            61     0.8786    0.3181 0.5444 0.1150

Structure-grouped folds, mean ± SD over the finished folds (n−1)   |   identifier-level folds (manuscript)
DualGATSynergy  PCC 0.315 ± 0.026 | AUC 0.608 | AUPR 0.266   |   PCC 0.346 ± 0.073 | AUC 0.656
RF              PCC 0.245 ± 0.006 | AUC 0.562 | AUPR 0.238   |   PCC 0.261 ± 0.058
XGB             PCC 0.193 ± 0.021 | AUC 0.571 | AUPR 0.212   |   PCC 0.207 ± 0.039
RF+cell         PCC 0.267 ± 0.015 | AUC 0.581 | AUPR 0.247   |   PCC 0.282 ± 0.066 | AUC 0.636
XGB+cell        PCC 0.320